In [2]:
import numpy as np
import matplotlib.pyplot as plt

G = 1
m=np.array([1,1,1])
#already calculated, working initial conditions
r=np.array([[-1.0,0.0,0.0],[1.0,0.0,0.0],[0.0,0.0,0.0]])
v=np.array([[0.464445,0.396060,0.0],[0.464445,0.396060,0.0],[-2*0.464445,-2*0.396060,0.0]])

The leapfrog method (second order)

In [4]:
#using the initial conditions above
r_l=r.copy()
v_l=v.copy()
#velocity correction so that the centre of mass o fthe system stays at one point -> plot and read
p = m[0]*v_l[0] + m[1]*v_l[1] + m[2]*v_l[2]
vcm = p / (m[0] + m[1] + m[2])    # centre of mass velocity
v_l = v_l - vcm

#∆t
dt = 0.001

#outer acceleration function!!!
def accelerations(r, m):
    a = np.zeros_like(r)
    for j in range(len(m)):
        for k in range(j + 1, len(m)):
            d = r[k] - r[j]
            dcube = (np.linalg.norm(d))**3
            a[j] += G * m[k] * (d/dcube)
            a[k] -= G * m[j] * (d/dcube)
    return a

a = accelerations(r_l,m)

#defining energy for energy conservation and erro rplot   (E = kin-pot)
def energy(r, v, m):
    kin = 0.0
    for j in range(len(m)):     #kinetic energy
        kin += 1/2 * m[j] * (np.linalg.norm(v[j]))**2

    pot = 0.0
    for j in range(len(m)):   #potntial energy 
        for k in range(j + 1, len(m)):
            pot += G * (m[j]*m[k]) / np.linalg.norm(r[k] - r[j])

    ener = kin - pot 
    return ener


#storing data for the energy calculation later
positions = [r_l.copy()]
velocities = [v_l.copy()]

#main loop and KDK implementation (half kick -> drift -> half kick)
n_steps = 10000

for i in range(n_steps):
    v_l += a * dt / 2   #half kick I
    r_l += v_l *dt        #drift
    a = accelerations(r_l,m)    #new a due to new positions(acceleration only depends on position)
    v_l += a * dt / 2   #half kick II
    positions.append(r_l.copy())
    velocities.append(v_l.copy())

#turn positions & velocities to array to use for energy 
positions = np.array(positions)
velocities = np.array(velocities)

#energy loop for comparison and plot
energies = []
for i in range(n_steps + 1):
    En = energy(positions[i], velocities[i], m)
    energies.append(En)
energies = np.array(energies)    

#relative error
E0 = energies[0]
diff = energies - E0
e_rel = diff / E0
e_rel = np.abs(e_rel)



Euler method (first order)

In [5]:
#we use the same r, v, m, dt, accelerations, velocities, positions as in leapfrog -> comparison at the end   (..._e always corresponds to the euler part of the code)
r_e = r.copy()
v_e = v.copy()

a_e = accelerations(r_e, m)

#same velocity correction for clear plots and animation
p_e = m[0]*v_e[0] + m[1]*v_e[1] + m[2]*v_e[2]
vcm_e = p_e / (m[0] + m[1] + m[2])
v_e = v_e - vcm_e

#storing data for energy
positions_e = [r_e.copy()]
velocities_e = [v_e.copy()]

#loop and Euler method (accelerations for current position -> full drift -> full kick)
for i in range(n_steps):
    r_e += v_e * dt
    v_e += a_e * dt
    a_e = accelerations(r_e, m)
    positions_e.append(r_e.copy())
    velocities_e.append(v_e.copy())

#turn velocities_e and positions_e into array
positions_e = np.array(positions_e)
velocities_e = np.array(velocities_e)

#enery for comparison plot
energies_e = []
for i in range(n_steps + 1):
    En_e = energy(positions_e[i], velocities_e[i], m)
    energies_e.append(En_e)
energies_e = np.array(energies_e)

#relative energy error
E0_e = energies_e[0]
diff_e = energies_e - E0_e
e_rel_e = diff_e / E0_e
e_rel_e = np.abs(e_rel_e)


Runge-Kutta method (fourth order)

In [3]:
#variables
r_rk4=r.copy()
v_rk4=v.copy()
h=0.001

#function for computing the acceleration of each body
def acceleration(r,m,G):
    a=np.zeros_like(r)
    for i in range (3):
        for j in range (3):
            if(i!=j):
                a[i]+=-G*m[j]*(r[i]-r[j])/(np.linalg.norm(r[i]-r[j]))**3
    return a

#function for computing the next velocity and position vector using fourth order Runga-Kutta
def RK4(r,v,m,G,h):
    k1_r=v
    k1_v=acceleration(r,m,G)

    k2_r=v+h/2*k1_v
    k2_v=acceleration(r+h/2*k1_r,m,G)

    k3_r=v+h/2*k2_v
    k3_v=acceleration(r+h/2*k2_r,m,G)

    k4_r=v+h*k3_v
    k4_v=acceleration(r+h*k3_r,m,G)

    r_new=r+h/6*(k1_r+2*k2_r+2*k3_r+k4_r)
    v_new=v+h/6*(k1_v+2*k2_v+2*k3_v+k4_v)

    return r_new,v_new

steps=10000
#arrays for storing calculated datas
p=np.zeros((steps,3,3))
s=np.zeros((steps,3,3))

#function for calculating the data points
for step in range (steps):
    next_r,next_v=RK4(r_rk4,v_rk4,m,G,h)
    p[step]=next_r
    s[step]=next_v
    r_rk4=next_r
    v_rk4=next_v